<h1>DOCUMENT LOADER</h1>

In [3]:
from langchain_community.document_loaders import Docx2txtLoader
import os

def document_loader(folder_path):
    document = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.docx'):
            file_path = os.path.join(folder_path,file_name)

            loader = Docx2txtLoader(file_path)
            pages = loader.load()

            document.extend(pages)

    return document

<h1>TEXT SPLITTER</h1>

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def text_splitter(document):

    textSplitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap= 40
    )

    chunks = textSplitter.split_documents(document)

    return chunks

<h1>VECTOR DATABASE</h1>

In [31]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

def create_vector_db(chunks):
    
    embedding = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

    vectorDatabase = FAISS.from_documents(
        chunks,
        embedding
    )

    return vectorDatabase

<h1>KEYWORD SEARCH</h1>

In [ ]:
from rank_bm25 import BM25Okapi

class keyword_Search:
    def __init__(self,chunks):
        self.chunks = chunks

        tokenied_data = []
        for c in chunks:
            tokens = c.page_content.lower().split()
            tokenied_data.append(tokens)

        self.bm250 = BM25Okapi(tokenied_data)

    def search(self,query,k=5):
        query = query.lower().split()

        scores = self.bm250.get_scores(query)

        rerank = sorted(range(len(scores)),
                        key = lambda i:scores[i],
                        reverse = True)

        top_k = rerank[:k]

        result = []

        for r in top_k:
            result.append(self.chunks[r])

        return result

<h1>HYBRID SEARCH</h1>

In [16]:
class HYBRID_SEARCH:
    def __init__(self,vector_db,bm250_db):
        self.vector_db= vector_db
        self.bm250_db = bm250_db

    def search(self,query,k=5):

        vector_result = self.vector_db.similarity_search(query,k=k)
        bm250_result = self.bm250_db.search(query,k=k)

        combined_result = vector_result + bm250_result

        seen_content = set()
        unique_result = []

        for c in combined_result:
            if c.page_content not in seen_content:
                unique_result.append(c)
                seen_content.add(c.page_content)

        return unique_result




<h1>RERANKER</h1>

In [43]:
from sentence_transformers import CrossEncoder

class ReRanker:
    def __init__(self):
        self.model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def rank(self,query,document,k=5):

        pairs = []

        for c in document:
            pairs.append([
                query,
                c.page_content
            ])

        scores = self.model.predict(pairs)

        reranker = sorted(zip(scores,document),
                    key = lambda i:i[0],
                    reverse = True)
        top_answers = reranker[:k]

        result = []

        for index,doc in top_answers:
            result.append(doc)

        return result
        

<h1>PROMPT<h1>

In [44]:
from langchain_groq import ChatGroq
from  langchain_core.prompts import ChatPromptTemplate
def generate_answer(question,document):

    context = ''
    for c in document:
        context+= c.page_content
        context+= '\n\n'

    prompt = ChatPromptTemplate.from_template(
        '''
        Answer the query from the context.

        if the question is out of the context. just say,"Question out of context"


        context
        {context}

        question
        {question}

        '''
    )

    chat_model = ChatGroq(model = 'openai/gpt-oss-20b',
                          temperature = 0)

    chain = prompt | chat_model

    response = chain.invoke({
        'context':context,
        'question':question
    }) 

    return response.content


<h1>PIPELINE</h1>

In [51]:
folder_path = 'E:\GEN-AI-PROJECTS'

import gradio as gr

#document loader
document = document_loader(folder_path)

# text splitter
chunks = text_splitter(document)

# create vector db
vector_db = create_vector_db(chunks)

# create keywprd search
keywordSearch = keyword_Search(chunks)

#hybrid retriveal
hybrd = HYBRID_SEARCH(vector_db,keywordSearch)

#reranker
reranker = ReRanker()

# create the answer
def create_answer(query,k=5):

    retrived_documents = hybrd.search(query,k=k)
    ranked_answer = reranker.rank(query,retrived_documents,k=k)

    answer = generate_answer(query,ranked_answer)

    sources = ""
    
    for i,doc in enumerate(retrived_documents):
        file_source = doc.metadata.get(
            'source',
            'unknown'
        )
        sources += f'\nSource {i+1}: {file_source}'

    final_response = answer
    final_response += '\n\nSources'
    final_response += sources

    return final_response

demo = gr.Interface(
    fn = create_answer,
    inputs=gr.Textbox(
        label = 'Enter your question'
    ),
    outputs=gr.Textbox(
        label= 'output',
        lines = 15
    ),
    title = 'ADVANCE RAG ARCHITECTURE'
)


demo.launch(share=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7868

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
